# Block 4 — the Hubbard atom from partial spectral functions

Numerical companion to

> F. B. Kugler, S.-S. B. Lee and J. von Delft,
> *Multipoint Correlation Functions: Spectral Representation and Numerical Evaluation*,
> [Phys. Rev. X **11**, 041006 (2021)](https://doi.org/10.1103/PhysRevX.11.041006)

The plan is to assemble the Matsubara correlators of the half-filled Hubbard atom
directly from their spectral representation, and to check them against the closed
forms the paper quotes:

| Step | Content | Benchmark |
|:--|:--|:--|
| **1** | spectrum: $H$, $d_\sigma$, $d^\dagger_\sigma$, $n_\sigma$; diagonalise; $\rho_i = e^{-\beta E_i}/Z$ | $Z = 2(1+e^{\beta u})$ by hand |
| 2 | partial spectral functions, Eq. (28), general in $\ell$ | — |
| 3 | the kernel, Eq. (46), two-branch form | finite by construction |
| 4 | $\ell = 2$: assemble Eq. (39) | $G(i\nu) = \tfrac12\sum_\pm (i\nu \pm U/2)^{-1}$, App. E |
| 5 | $\ell = 4$: 24 permutations, subtract Eq. (73), amputate Eq. (76) | Eqs. (85a), (85b) |
| 6 | null test at $U = 0$ | every connected object vanishes |

**This notebook currently implements Step 1 only.** Steps 2–6 are stubbed below in
the order they will be filled in.

Model: $H = \varepsilon_d (n_\uparrow + n_\downarrow) + U n_\uparrow n_\downarrow$
at $\varepsilon_d = -U/2$, on the four states
$|0\rangle, |\uparrow\rangle, |\downarrow\rangle, |\uparrow\downarrow\rangle$.
This is the $U/\Delta \to \infty$ limit of the symmetric AIM, Eq. (79).

In [1]:
include(joinpath(@__DIR__, "..", "src", "HubbardAtom.jl"))
using .HubbardAtom
using LinearAlgebra
using Printf

## Step 1 — spectrum

Build the operators with correct fermionic signs, diagonalise $H$, and form the
Boltzmann weights. Everything later hangs off the eigenbasis produced here:
Eq. (28) needs the matrix elements $(O)_{i,i+1}$ in the energy eigenbasis, which
`to_eigenbasis` supplies.

The operators come from a Jordan–Wigner transform with fermionic mode order
$(\uparrow, \downarrow)$,

$$ d_\uparrow = \mathbb{1} \otimes f, \qquad d_\downarrow = f \otimes z, \qquad
   f = \begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}, \quad
   z = \begin{pmatrix} 1 & 0 \\ 0 & -1\end{pmatrix}, $$

so the parity string $z$ over the $\uparrow$ mode is what produces
$d_\downarrow |\uparrow\downarrow\rangle = -|\uparrow\rangle$. That sign is not
cosmetic: it feeds every permutation sign $\zeta_p$ in Eq. (39).

In [2]:
U, β = 2.0, 5.0
model = HubbardAtomModel(U, β)
ops = operators(model)

@printf("U = %.3f,  β = %.3f,  εd = %.3f,  u = U/2 = %.3f\n",
        model.U, model.β, level_position(model), half_interaction(model))
println("\nFock basis order: ", join(BASIS_LABELS, "  "))
println("\nH (Fock basis) =")
display(ops.H)

U = 2.000,  β = 5.000,  εd = -1.000,  u = U/2 = 1.000

Fock basis order: 

|0⟩  |↑⟩  |↓⟩  |↑↓⟩

H (Fock basis) =


4×4 Matrix{Float64}:
 0.0   0.0   0.0  0.0
 0.0  -1.0   0.0  0.0
 0.0   0.0  -1.0  0.0
 0.0   0.0   0.0  0.0

In [3]:
# Fermionic algebra: the checks that catch a wrong Jordan-Wigner sign.
I4 = Matrix(1.0I, DIM, DIM)

println("{d↑, d†↑} == 1        : ", ops.d_up * ops.dag_up + ops.dag_up * ops.d_up ≈ I4)
println("{d↓, d†↓} == 1        : ", ops.d_dn * ops.dag_dn + ops.dag_dn * ops.d_dn ≈ I4)
println("{d↑, d†↓} == 0        : ", ops.d_up * ops.dag_dn + ops.dag_dn * ops.d_up ≈ zeros(DIM, DIM))
println("{d↑, d↓}  == 0        : ", ops.d_up * ops.d_dn + ops.d_dn * ops.d_up ≈ zeros(DIM, DIM))
println("d↑² == d↓² == 0       : ", ops.d_up^2 ≈ zeros(DIM, DIM) && ops.d_dn^2 ≈ zeros(DIM, DIM))
println("[H, N] == 0           : ", ops.H * ops.N ≈ ops.N * ops.H)
println("[H, Sz] == 0          : ", ops.H * ops.Sz ≈ ops.Sz * ops.H)

# the fermionic sign itself
up_dn = [0.0, 0.0, 0.0, 1.0]
up    = [0.0, 1.0, 0.0, 0.0]
println("\nd↓|↑↓⟩ = -|↑⟩         : ", ops.d_dn * up_dn ≈ -up)
println("d↑|↑↓⟩ = +|↓⟩         : ", ops.d_up * up_dn ≈ [0.0, 0.0, 1.0, 0.0])

{d↑, d†↑} == 1        : true


{d↓, d†↓} == 1        : true
{d↑, d†↓} == 0        : true
{d↑, d↓}  == 0        : true
d↑² == d↓² == 0       : true


[H, N] == 0           : true
[H, Sz] == 0          : true

d↓|↑↓⟩ = -|↑⟩         : 

true
d↑|↑↓⟩ = +|↓⟩         : true


In [4]:
sp = spectrum(model)

println("eigenenergies   E = ", sp.E)
println("closed form       = ", eigenenergies_analytic(model))
println("agree             : ", sp.E ≈ eigenenergies_analytic(model))

println("\nH·V == V·diag(E)  : ", ops.H * sp.V ≈ sp.V * Diagonal(sp.E))
println("V orthonormal     : ", permutedims(sp.V) * sp.V ≈ Matrix(1.0I, DIM, DIM))

eigenenergies   E = 

[-1.0, -1.0, 0.0, 0.0]
closed form       = 

[-1.0, -1.0, 0.0, 0.0]
agree             : true

H·V == V·diag(E)  : true
V orthonormal     : true


In [5]:
# Check Z by hand.  With u = U/2 and εd = -U/2 the spectrum is
#     E(|0⟩) = 0,  E(|↑⟩) = E(|↓⟩) = -u,  E(|↑↓⟩) = 2εd + U = 0,
# so     Z = 1 + e^{βu} + e^{βu} + 1 = 2(1 + e^{βu}).
Z_hand = partition_function_analytic(model)

@printf("Z (diagonalisation) = %.15g\n", sp.Z)
@printf("Z (by hand)         = %.15g\n", Z_hand)
@printf("relative difference = %.3e\n", abs(sp.Z - Z_hand) / Z_hand)
println("agree to machine precision: ", isapprox(sp.Z, Z_hand; rtol = 1e-12))

Z (diagonalisation) = 298.826318205153
Z (by hand)         = 298.826318205153


relative difference = 1.902e-16
agree to machine precision: 

true


In [6]:
# Boltzmann weights and the quantum numbers of each eigenstate.
Nvals, Szvals = quantum_numbers(sp, ops)

println(" i      E_i        ρ_i          N      Sz")
println("-"^50)
for i in 1:DIM
    @printf("%2d  %9.5f  %11.4e  %5.1f  %6.1f\n", i, sp.E[i], sp.ρ[i], Nvals[i], Szvals[i])
end
@printf("\nΣ ρ_i = %.15g\n", sum(sp.ρ))

 i      E_i        ρ_i          N      Sz
--------------------------------------------------
 1   -1.00000   4.9665e-01    1.0    -0.5
 2   -1.00000   4.9665e-01    1.0     0.5


 3    0.00000   3.3464e-03    0.0     0.0
 4    0.00000   3.3464e-03    2.0     0.0

Σ ρ_i = 1


In [7]:
# Particle-hole symmetry at εd = -U/2 pins the atom to half filling.
@printf("⟨n↑⟩       = %.15g\n", thermal_average(sp, ops.n_up))
@printf("⟨n↓⟩       = %.15g\n", thermal_average(sp, ops.n_dn))
@printf("⟨N⟩        = %.15g\n", thermal_average(sp, ops.N))
@printf("⟨Sz⟩       = %.3e\n",  thermal_average(sp, ops.Sz))
@printf("⟨n↑n↓⟩     = %.15g   (exact: 1/Z = %.15g)\n",
        thermal_average(sp, ops.n_up * ops.n_dn), 1 / sp.Z)
@printf("⟨H⟩        = %.15g\n", thermal_average(sp, ops.H))

# tanh(βu/2) of Eq. (85) is this spectrum's polarisation
u = half_interaction(model)
singly = 2 * exp(β * u) / Z_hand
@printf("\n2·P(singly occupied) - 1 = %.15g\ntanh(βu/2)               = %.15g\n",
        2 * singly - 1, tanh(β * u / 2))

⟨n↑⟩       = 0.5
⟨n↓⟩       = 0.5
⟨N⟩        = 1
⟨Sz⟩       = 0.000e+00
⟨n↑n↓⟩     = 0.00334642546214243   (exact: 1/Z = 0.00334642546214243)
⟨H⟩        = -0.993307149075715



2·P(singly occupied) - 1 = 0.98661429815143
tanh(βu/2)               = 0.98661429815143


In [8]:
# What Step 2 will consume: operators in the energy eigenbasis, Eq. (28).
d_rot = to_eigenbasis(sp, ops.d_up)
println("d↑ in the energy eigenbasis:")
display(round.(d_rot, digits = 12))
println("\nnonzero elements (i, j, value), each lowering N by one:")
for j in 1:DIM, i in 1:DIM
    if abs(d_rot[i, j]) > 1e-10
        @printf("  (%d, %d)  % .6f    N: %.1f → %.1f\n",
                i, j, d_rot[i, j], Nvals[j], Nvals[i])
    end
end

d↑ in the energy eigenbasis:


4×4 Matrix{Float64}:
 0.0  0.0  0.0  1.0
 0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0
 0.0  0.0  0.0  0.0


nonzero elements (i, j, value), each lowering N by one:
  (3, 2)   1.000000    N: 1.0 → 0.0
  (1, 4)   1.000000    N: 2.0 → 1.0


In [9]:
# Z against its closed form across a parameter sweep, including U = 0 and U < 0.
println("    U      β        Z (diag)          Z (hand)        rel.err")
println("-"^66)
for (Ui, βi) in [(1.0, 1.0), (2.0, 5.0), (4.0, 0.3), (0.0, 2.0), (8.0, 10.0), (-2.0, 1.5)]
    mi = HubbardAtomModel(Ui, βi)
    spi = spectrum(mi)
    Zi = partition_function_analytic(mi)
    @printf("%6.1f %6.1f  %16.8e  %16.8e  %10.2e\n",
            Ui, βi, spi.Z, Zi, abs(spi.Z - Zi) / Zi)
end

    U      β        Z (diag)          Z (hand)        rel.err
------------------------------------------------------------------
   1.0    1.0    5.29744254e+00    5.29744254e+00    1.68e-16
   2.0    5.0    2.98826318e+02    2.98826318e+02    1.90e-16
   4.0    0.3    5.64423760e+00    5.64423760e+00    0.00e+00
   0.0    2.0    4.00000000e+00    4.00000000e+00    0.00e+00
   8.0   10.0    4.70770534e+17    4.70770534e+17    0.00e+00
  -2.0    1.5    2.44626032e+00    2.44626032e+00    0.00e+00


## Step 2 — partial spectral functions *(not yet implemented)*

Implement Eq. (28) literally, returning a list of `(position, weight)` pairs and
kept general in $\ell$ from the start, so the one function serves
$\ell = 2, 3, 4$:

$$ S[\mathcal{O}](\omega'_p) = \sum_{1 \cdots \ell} \rho_1
   \Big[ \prod_{i=1}^{\ell-1} (O_{\bar{i}})_{i,i+1}\,
   \delta\big(\omega'_{\bar{1}\cdots\bar{i}} - E_{i+1,1}\big) \Big]
   (O_{\bar{\ell}})_{\ell,1}, \qquad E_{i+1,1} = E_{i+1} - E_1 . $$

The matrix elements are `to_eigenbasis(sp, O)` from Step 1, and $\rho_1$ is `sp.ρ`.
Because the PSFs are sums of $\delta$'s, the $\omega'$ integral in Eq. (39) is a
finite sum — no grids, no broadening.

## Step 3 — the kernel *(not yet implemented)*

Implement Eq. (46), the two-branch form, with an explicit test for a vanishing
composite $\Omega_{\bar{1}\cdots\bar{j}}$:

$$ K(\Omega_p) = \begin{cases}
\displaystyle\prod_{i=1}^{\ell-1} \Omega^{-1}_{\bar{1}\cdots\bar{i}},
& \text{if } \prod_{i=1}^{\ell-1} \Omega_{\bar{1}\cdots\bar{i}} \neq 0, \\[2ex]
-\dfrac{1}{2}\Big[\beta + \displaystyle\sum_{\substack{i=1 \\ i \neq j}}^{\ell-1}
\Omega^{-1}_{\bar{1}\cdots\bar{i}}\Big]
\displaystyle\prod_{\substack{i=1 \\ i \neq j}}^{\ell-1}
\Omega^{-1}_{\bar{1}\cdots\bar{i}},
& \text{if } \Omega_{\bar{1}\cdots\bar{j}} = 0,
\end{cases} $$

with $\Omega_{\bar{1}\cdots\bar{i}} = i\omega_{\bar{1}\cdots\bar{i}} - \omega'_{\bar{1}\cdots\bar{i}}$.

Deliberately **not** Eq. (45) with an $\epsilon$-regulator: Eq. (46) is finite by
construction, and the paper notes the two are equivalent with the latter more
convenient numerically, being manifestly free from divergences.

## Step 4 — first milestone, $\ell = 2$ *(not yet implemented)*

Assemble Eq. (39),

$$ G(i\omega) = \sum_p \zeta_p \int d^{\ell-1}\omega'_p \;
   K(i\omega_p - \omega'_p)\, S[\mathcal{O}_p](\omega'_p) $$

for $\mathcal{O} = (d_\sigma, d^\dagger_\sigma)$, and compare against the exact
propagator of Appendix E,

$$ G(i\nu) = \frac{1}{2}\sum_{\pm} (i\nu \pm U/2)^{-1}, \qquad
   \Sigma(i\nu) = \frac{U^2}{4 i\nu}. $$

**Stop rule:** do not proceed past this until it matches to machine precision.

## Step 5 — $\ell = 4$ *(not yet implemented)*

Take $\mathcal{O} = (d_\sigma, d^\dagger_\sigma, d_{\sigma'}, d^\dagger_{\sigma'})$
over all 24 permutations. Subtract the disconnected part per Eq. (73), amputate the
external legs per Eq. (76) — using the imaginary shifts of Appendix D, without which
$F$ comes out wrong — and compare against Eqs. (85a) and (85b) for both
$\sigma\sigma' = \uparrow\downarrow$ and $\uparrow\uparrow$:

$$ F_{\uparrow\downarrow} = 2u + u^3 \frac{\sum_i (i\omega_i)^2}{\prod_i (i\omega_i)}
   - \frac{6u^5}{\prod_i (i\omega_i)}
   + \beta u^2 \frac{\delta_{\omega_{12}} \mathrm{th}
   + \delta_{\omega_{13}}(\mathrm{th}-1)
   + \delta_{\omega_{14}}(\mathrm{th}+1)}
   {\prod_i (i\omega_i + u)\prod_i (i\omega_i)}, $$

$$ F_{\uparrow\uparrow} = \beta u^2
   \frac{\delta_{\omega_{14}} - \delta_{\omega_{12}}}
   {\prod_i (i\omega_i + u)\prod_i (i\omega_i)}, $$

with $u = U/2$, $\mathrm{th} = \tanh(\beta u / 2)$, $i \in \{1,2,3,4\}$.

If this fails *only* in the $\beta\delta_\omega$ terms, diff against Eqs. (B23)–(B28).

## Step 6 — null test *(not yet implemented)*

Set $U = 0$. Every connected object must vanish identically. This exercises all 24
permutations and every sign $\zeta_p$ against a target of exactly zero.